# ResumeIQ — Notebook 03: Model Exploration & Embedding Generation

In this notebook, we explore the deep learning embedding pipeline used in **ResumeIQ**:
1. **Hugging Face Transformer (`sentence-transformers/bert-base-nli-mean-tokens`)**
2. **Tokenization, Attention Masks, and Attention-Weighted Mean Pooling**
3. **Doc2Vec Dense Vectorization**

## 1. Import Modules and Setup

In [ ]:
from pathlib import Path
import sys
import torch
import numpy as np

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import (
    load_bert_model_and_tokenizer,
    get_HF_embeddings,
    mean_pooling,
    get_doc2vec_embeddings,
    cosine
)
print("Model modules imported successfully!")

## 2. BERT Architecture & Tokenization
Let's load the pretrained BERT tokenizer and model to understand the internal tensor transformations.

In [ ]:
tokenizer, model = load_bert_model_and_tokenizer()
print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Model: {model.__class__.__name__}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Tokenization & Attention Mask Breakdown

In [ ]:
sample_text = "Machine Learning Engineer with Python and PyTorch."
encoded = tokenizer(sample_text, padding=True, truncation=True, return_tensors="pt", max_length=512)

print("Input IDs (token indexes):", encoded["input_ids"].shape)
print("Tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))
print("Attention Mask:", encoded["attention_mask"])

## 4. Attention-Weighted Mean Pooling

BERT produces a representation $\mathbf{h}_i \in \mathbb{R}^{768}$ for each token $i$.
To produce a single fixed-length document vector $\mathbf{u}$, we calculate the weighted average ignoring padding:

$$\mathbf{u} = \frac{\sum_{i=1}^N \mathbf{h}_i \cdot m_i}{\sum_{i=1}^N m_i}$$

where $m_i \in \{0, 1\}$ is the attention mask.

In [ ]:
with torch.no_grad():
    model_output = model(**encoded)

raw_tokens = model_output[0]
print("Token embeddings tensor shape:", raw_tokens.shape)  # [batch_size, seq_len, 768]

pooled_embedding = mean_pooling(model_output, encoded["attention_mask"])
print("Pooled document embedding shape:", pooled_embedding.shape)  # [batch_size, 768]
print("Embedding vector (first 5 values):", pooled_embedding[0][:5].tolist())

## 5. Doc2Vec Embeddings Alternative
ResumeIQ also includes a Doc2Vec engine that learns document embeddings directly from word sequences.

In [ ]:
jd_text = "Looking for a Senior Python and NLP developer."
resume_list = ["Python developer with NLP experience.", "React and CSS web developer."]

jd_d2v, res_d2v = get_doc2vec_embeddings(jd_text, resume_list)
print("Doc2Vec JD vector shape:", jd_d2v.shape)
print("Doc2Vec Resume 1 vector shape:", res_d2v[0].shape)

scores = cosine(res_d2v, jd_d2v)
for idx, score in enumerate(scores):
    print(f"Resume {idx+1} Doc2Vec Cosine Match: {score}%")